In [ ]:
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'README.md').exists() and (p / 'data').exists():
            return p
    raise FileNotFoundError('Project root not found. Make sure README.md and data/ exist.')

PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'figures'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_DIR =', RAW_DIR)
print('PROCESSED_DIR =', PROCESSED_DIR)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd

hypno_df = pd.read_csv(PROCESSED_DIR / '../data/processed/hypno_df.csv')
features_df = extract_sleep_features(hypno_df, subject_id='subject_001')
features_df 

def clean_hypnogram(df):
    df = df.copy()
    df = df[~df['description'].str.contains('?', regex=False, na=False)].copy()
    df = df[df['duration'] > 0].copy()
    df = df.reset_index(drop=True)
    return df
    def find_sleep_boundaries(df):
    sleep_mask = df['description'] != 'Sleep stage W'

    if sleep_mask.sum() == 0:
        return None, None

    first_sleep_idx = df[sleep_mask].index[0]
    last_sleep_idx = df[sleep_mask].index[-1]

    return first_sleep_idx, last_sleep_idx

    def extract_sleep_features(hypno_df, subject_id=None):
    df = clean_hypnogram(hypno_df)

    total_recording_time_sec = df['duration'].sum()

    first_sleep_idx, last_sleep_idx = find_sleep_boundaries(df)

    if first_sleep_idx is None:
        return pd.DataFrame([{
            'subject_id': subject_id,
            'recording_hours': total_recording_time_sec / 3600,
            'sleep_period_hours': np.nan,
            'total_sleep_hours': 0,
            'sleep_latency_min': np.nan,
            'rem_latency_min': np.nan,
            'fragmentation': np.nan,
            'wake_pct_in_sleep_period': np.nan,
            'rem_pct_of_tst': np.nan,
            'n1_pct_of_tst': np.nan,
            'n2_pct_of_tst': np.nan,
            'deep_sleep_pct_of_tst': np.nan,
            'sleep_efficiency': np.nan
        }])

    sleep_latency_sec = df.loc[:first_sleep_idx - 1, 'duration'].sum() if first_sleep_idx > 0 else 0

    rem_indices = df[df['description'] == 'Sleep stage R'].index
    rem_after_sleep = rem_indices[rem_indices >= first_sleep_idx]

    if len(rem_after_sleep) > 0:
        first_rem_idx = rem_after_sleep[0]
        rem_latency_sec = df.loc[first_sleep_idx:first_rem_idx - 1, 'duration'].sum() if first_rem_idx > first_sleep_idx else 0
    else:
        rem_latency_sec = np.nan

    sleep_period_df = df.loc[first_sleep_idx:last_sleep_idx].copy().reset_index(drop=True)
    sleep_period_time_sec = sleep_period_df['duration'].sum()

    wake_in_sleep_period_sec = sleep_period_df.loc[
        sleep_period_df['description'] == 'Sleep stage W', 'duration'
    ].sum()

    total_sleep_time_sec = sleep_period_time_sec - wake_in_sleep_period_sec

    sleep_only_df = sleep_period_df[sleep_period_df['description'] != 'Sleep stage W'].copy()
    sleep_stage_time = sleep_only_df.groupby('description')['duration'].sum()

    if total_sleep_time_sec > 0:
        sleep_stage_pct = sleep_stage_time / total_sleep_time_sec * 100
    else:
        sleep_stage_pct = pd.Series(dtype=float)

    fragmentation = (sleep_period_df['description'] != sleep_period_df['description'].shift()).sum()

    features = {
        'subject_id': subject_id,
        'recording_hours': total_recording_time_sec / 3600,
        'sleep_period_hours': sleep_period_time_sec / 3600,
        'total_sleep_hours': total_sleep_time_sec / 3600,
        'sleep_latency_min': sleep_latency_sec / 60,
        'rem_latency_min': rem_latency_sec / 60 if pd.notna(rem_latency_sec) else np.nan,
        'fragmentation': fragmentation,
        'wake_pct_in_sleep_period': (
            wake_in_sleep_period_sec / sleep_period_time_sec * 100
            if sleep_period_time_sec > 0 else np.nan
        ),
        'rem_pct_of_tst': sleep_stage_pct.get('Sleep stage R', 0),
        'n1_pct_of_tst': sleep_stage_pct.get('Sleep stage 1', 0),
        'n2_pct_of_tst': sleep_stage_pct.get('Sleep stage 2', 0),
        'deep_sleep_pct_of_tst': (
            sleep_stage_pct.get('Sleep stage 3', 0) +
            sleep_stage_pct.get('Sleep stage 4', 0)
        ),
        'sleep_efficiency': (
            total_sleep_time_sec / sleep_period_time_sec
            if sleep_period_time_sec > 0 else np.nan
        )
    }

    return pd.DataFrame([features])

    def extract_sleep_features(hypno_df, subject_id=None):
    df = clean_hypnogram(hypno_df)

    total_recording_time_sec = df['duration'].sum()

    first_sleep_idx, last_sleep_idx = find_sleep_boundaries(df)

    if first_sleep_idx is None:
        return pd.DataFrame([{
            'subject_id': subject_id,
            'recording_hours': total_recording_time_sec / 3600,
            'sleep_period_hours': np.nan,
            'total_sleep_hours': 0,
            'sleep_latency_min': np.nan,
            'rem_latency_min': np.nan,
            'fragmentation': np.nan,
            'wake_pct_in_sleep_period': np.nan,
            'rem_pct_of_tst': np.nan,
            'n1_pct_of_tst': np.nan,
            'n2_pct_of_tst': np.nan,
            'deep_sleep_pct_of_tst': np.nan,
            'sleep_efficiency': np.nan
        }])

    sleep_latency_sec = df.loc[:first_sleep_idx - 1, 'duration'].sum() if first_sleep_idx > 0 else 0

    rem_indices = df[df['description'] == 'Sleep stage R'].index
    rem_after_sleep = rem_indices[rem_indices >= first_sleep_idx]

    if len(rem_after_sleep) > 0:
        first_rem_idx = rem_after_sleep[0]
        rem_latency_sec = df.loc[first_sleep_idx:first_rem_idx - 1, 'duration'].sum() if first_rem_idx > first_sleep_idx else 0
    else:
        rem_latency_sec = np.nan

    sleep_period_df = df.loc[first_sleep_idx:last_sleep_idx].copy().reset_index(drop=True)
    sleep_period_time_sec = sleep_period_df['duration'].sum()

    wake_in_sleep_period_sec = sleep_period_df.loc[
        sleep_period_df['description'] == 'Sleep stage W', 'duration'
    ].sum()

    total_sleep_time_sec = sleep_period_time_sec - wake_in_sleep_period_sec

    sleep_only_df = sleep_period_df[sleep_period_df['description'] != 'Sleep stage W'].copy()
    sleep_stage_time = sleep_only_df.groupby('description')['duration'].sum()

    if total_sleep_time_sec > 0:
        sleep_stage_pct = sleep_stage_time / total_sleep_time_sec * 100
    else:
        sleep_stage_pct = pd.Series(dtype=float)

    fragmentation = (sleep_period_df['description'] != sleep_period_df['description'].shift()).sum()

    features = {
        'subject_id': subject_id,
        'recording_hours': total_recording_time_sec / 3600,
        'sleep_period_hours': sleep_period_time_sec / 3600,
        'total_sleep_hours': total_sleep_time_sec / 3600,
        'sleep_latency_min': sleep_latency_sec / 60,
        'rem_latency_min': rem_latency_sec / 60 if pd.notna(rem_latency_sec) else np.nan,
        'fragmentation': fragmentation,
        'wake_pct_in_sleep_period': (
            wake_in_sleep_period_sec / sleep_period_time_sec * 100
            if sleep_period_time_sec > 0 else np.nan
        ),
        'rem_pct_of_tst': sleep_stage_pct.get('Sleep stage R', 0),
        'n1_pct_of_tst': sleep_stage_pct.get('Sleep stage 1', 0),
        'n2_pct_of_tst': sleep_stage_pct.get('Sleep stage 2', 0),
        'deep_sleep_pct_of_tst': (
            sleep_stage_pct.get('Sleep stage 3', 0) +
            sleep_stage_pct.get('Sleep stage 4', 0)
        ),
        'sleep_efficiency': (
            total_sleep_time_sec / sleep_period_time_sec
            if sleep_period_time_sec > 0 else np.nan
        )
    }

    return pd.DataFrame([features])

    hypno_df = pd.read_csv(PROCESSED_DIR / '../data/processed/hypno_df.csv')
features_df = extract_sleep_features(hypno_df, subject_id='subject_001')
features_df

features_df.to_csv(PROCESSED_DIR / 'sleep_features_subject_001.csv', index=False)
print('Saved:', PROCESSED_DIR / 'sleep_features_subject_001.csv')

In [ ]:
def clean_hypnogram(df):
    df = df.copy()

    df = df[~df['description'].str.contains('?', regex=False, na=False)].copy()

    df = df[df['duration'] > 0].copy()

    df = df.reset_index(drop=True)

    return df

In [ ]:
def find_sleep_boundaries(df):
    sleep_mask = df['description'] != 'Sleep stage W'

    if sleep_mask.sum() == 0:
        return None, None

    first_sleep_idx = df[sleep_mask].index[0]
    last_sleep_idx = df[sleep_mask].index[-1]

    return first_sleep_idx, last_sleep_idx

In [ ]:
def extract_sleep_features(hypno_df, subject_id=None):
    df = clean_hypnogram(hypno_df)

    total_recording_time_sec = df['duration'].sum()

    first_sleep_idx, last_sleep_idx = find_sleep_boundaries(df)

    if first_sleep_idx is None:
        return pd.DataFrame([{
            'subject_id': subject_id,
            'recording_hours': total_recording_time_sec / 3600,
            'sleep_period_hours': np.nan,
            'total_sleep_hours': 0,
            'sleep_latency_min': np.nan,
            'rem_latency_min': np.nan,
            'fragmentation': np.nan,
            'wake_pct_in_sleep_period': np.nan,
            'rem_pct_of_tst': np.nan,
            'n1_pct_of_tst': np.nan,
            'n2_pct_of_tst': np.nan,
            'deep_sleep_pct_of_tst': np.nan,
            'sleep_efficiency': np.nan
        }])

    sleep_latency_sec = df.loc[:first_sleep_idx - 1, 'duration'].sum() if first_sleep_idx > 0 else 0

    rem_indices = df[df['description'] == 'Sleep stage R'].index
    rem_after_sleep = rem_indices[rem_indices >= first_sleep_idx]

    if len(rem_after_sleep) > 0:
        first_rem_idx = rem_after_sleep[0]
        rem_latency_sec = df.loc[first_sleep_idx:first_rem_idx - 1, 'duration'].sum() if first_rem_idx > first_sleep_idx else 0
    else:
        rem_latency_sec = np.nan

    sleep_period_df = df.loc[first_sleep_idx:last_sleep_idx].copy().reset_index(drop=True)

    sleep_period_time_sec = sleep_period_df['duration'].sum()

    wake_in_sleep_period_sec = sleep_period_df.loc[
        sleep_period_df['description'] == 'Sleep stage W', 'duration'
    ].sum()

    total_sleep_time_sec = sleep_period_time_sec - wake_in_sleep_period_sec

    sleep_only_df = sleep_period_df[sleep_period_df['description'] != 'Sleep stage W'].copy()
    sleep_stage_time = sleep_only_df.groupby('description')['duration'].sum()

    if total_sleep_time_sec > 0:
        sleep_stage_pct = sleep_stage_time / total_sleep_time_sec * 100
    else:
        sleep_stage_pct = pd.Series(dtype=float)

    fragmentation = (sleep_period_df['description'] != sleep_period_df['description'].shift()).sum()

    features = {
        'subject_id': subject_id,
        'recording_hours': total_recording_time_sec / 3600,
        'sleep_period_hours': sleep_period_time_sec / 3600,
        'total_sleep_hours': total_sleep_time_sec / 3600,
        'sleep_latency_min': sleep_latency_sec / 60,
        'rem_latency_min': rem_latency_sec / 60 if pd.notna(rem_latency_sec) else np.nan,
        'fragmentation': fragmentation,
        'wake_pct_in_sleep_period': (
            wake_in_sleep_period_sec / sleep_period_time_sec * 100
            if sleep_period_time_sec > 0 else np.nan
        ),
        'rem_pct_of_tst': sleep_stage_pct.get('Sleep stage R', 0),
        'n1_pct_of_tst': sleep_stage_pct.get('Sleep stage 1', 0),
        'n2_pct_of_tst': sleep_stage_pct.get('Sleep stage 2', 0),
        'deep_sleep_pct_of_tst': (
            sleep_stage_pct.get('Sleep stage 3', 0) +
            sleep_stage_pct.get('Sleep stage 4', 0)
        ),
        'sleep_efficiency': (
            total_sleep_time_sec / sleep_period_time_sec
            if sleep_period_time_sec > 0 else np.nan
        )
    }

    return pd.DataFrame([features])

In [ ]:
features_df = extract_sleep_features(hypno_df, subject_id='subject_001')
features_df

In [ ]:
print(type(hypno_df))

In [ ]:
%whos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

hypno_df = pd.read_csv(PROCESSED_DIR / '../data/processed/hypno_df.csv')
hypno_df.head()

In [ ]:
features_df = extract_sleep_features(hypno_df, subject_id='subject_001')
features_df

In [ ]:
features_df = extract_sleep_features(hypno_df, subject_id='subject_001')
features_df

In [ ]:
def print_sleep_feature_summary(features_df):
    row = features_df.iloc[0]

    print("Feature Engineering Summary")
    print("---------------------------")
    print(f"Recording time: {row['recording_hours']:.2f} h")
    print(f"Sleep period: {row['sleep_period_hours']:.2f} h")
    print(f"Total sleep time: {row['total_sleep_hours']:.2f} h")
    print(f"Sleep latency: {row['sleep_latency_min']:.2f} min")
    print(f"REM latency: {row['rem_latency_min']:.2f} min" if pd.notna(row['rem_latency_min']) else "REM latency: NaN")
    print(f"Fragmentation: {int(row['fragmentation'])}")
    print(f"Wake % in sleep period: {row['wake_pct_in_sleep_period']:.2f}%")
    print(f"REM % of TST: {row['rem_pct_of_tst']:.2f}%")
    print(f"Deep sleep % of TST: {row['deep_sleep_pct_of_tst']:.2f}%")
    print(f"Sleep efficiency: {row['sleep_efficiency']:.2%}")

In [ ]:
print_sleep_feature_summary(features_df)

In [ ]:
def get_sleep_period_df(hypno_df):
    df = clean_hypnogram(hypno_df)
    first_sleep_idx, last_sleep_idx = find_sleep_boundaries(df)

    if first_sleep_idx is None:
        return df.iloc[0:0].copy()

    return df.loc[first_sleep_idx:last_sleep_idx].copy().reset_index(drop=True)

In [ ]:
def plot_sleep_period_hypnogram(hypno_df, title='Sleep Period Hypnogram'):
    df = get_sleep_period_df(hypno_df)

    if df.empty:
        print("No sleep period found.")
        return

    stage_map = {
        'Sleep stage W': 4,
        'Sleep stage R': 3,
        'Sleep stage 1': 2,
        'Sleep stage 2': 1,
        'Sleep stage 3': 0,
        'Sleep stage 4': 0
    }

    stage_labels = {
        4: 'Wake',
        3: 'REM',
        2: 'N1',
        1: 'N2',
        0: 'N3'
    }

    df = df[df['description'].isin(stage_map.keys())].copy()
    df['stage_num'] = df['description'].map(stage_map)

    x = []
    y = []
    current_time = 0

    for _, row in df.iterrows():
        start = current_time / 3600
        end = (current_time + row['duration']) / 3600
        stage = row['stage_num']

        x.extend([start, end])
        y.extend([stage, stage])

        current_time += row['duration']

    plt.figure(figsize=(14, 5))
    plt.step(x, y, where='post', linewidth=2)

    plt.yticks(list(stage_labels.keys()), [stage_labels[k] for k in stage_labels.keys()])
    plt.gca().invert_yaxis()
    plt.xlabel('Time (hours)')
    plt.ylabel('Sleep stage')
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_sleep_period_hypnogram(hypno_df, title='Subject 001 — Sleep Period Hypnogram')

In [ ]:
def plot_sleep_quality_radar(features_df, title='Sleep Quality Radar'):
    row = features_df.iloc[0]

    raw_metrics = {
        'Sleep duration': row['total_sleep_hours'],
        'Efficiency': row['sleep_efficiency'],
        'REM balance': row['rem_pct_of_tst'],
        'Deep sleep': row['deep_sleep_pct_of_tst'],
        'Latency': row['sleep_latency_min'],
        'Fragmentation': row['fragmentation'],
        'Wake in sleep period': row['wake_pct_in_sleep_period']
    }

    normalized = {
        'Sleep duration': min(raw_metrics['Sleep duration'] / 8, 1),  
        'Efficiency': min(raw_metrics['Efficiency'], 1),
        'REM balance': min(raw_metrics['REM balance'] / 25, 1),       
        'Deep sleep': min(raw_metrics['Deep sleep'] / 20, 1),         
        'Latency': 1 - min(raw_metrics['Latency'] / 60, 1),           
        'Fragmentation': 1 - min(raw_metrics['Fragmentation'] / 80, 1),
        'Wake in sleep period': 1 - min(raw_metrics['Wake in sleep period'] / 30, 1)
    }

    labels = list(normalized.keys())
    values = list(normalized.values())

    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    values += values[:1]
    angles += angles[:1]

    plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)

    ax.plot(angles, values, linewidth=2)
    ax.fill(angles, values, alpha=0.25)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels)

    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'])

    plt.title(title, pad=20)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_sleep_quality_radar(features_df, title='Subject 001 — Sleep Quality Profile')

In [ ]:
features_df.T

In [ ]:
all_features = []

for subject_id, subject_hypno_df in all_hypnograms.items():
    subj_features = extract_sleep_features(subject_hypno_df, subject_id=subject_id)
    all_features.append(subj_features)

sleep_features_df = pd.concat(all_features, ignore_index=True)
sleep_features_df.head()

In [ ]:

features_df = extract_sleep_features(hypno_df, subject_id='subject_001')

print_sleep_feature_summary(features_df)

plot_sleep_period_hypnogram(
    hypno_df,
    title='Subject 001 — Sleep Period Hypnogram'
)

plot_sleep_quality_radar(
    features_df,
    title='Subject 001 — Sleep Quality Profile'
)

features_df.to_csv(PROCESSED_DIR / 'sleep_features_subject_001.csv', index=False)

print("\nSaved file: sleep_features_subject_001.csv")

features_df

## Feature engineering conclusion

In this notebook, the raw hypnogram was transformed into subject-level sleep features suitable for downstream machine learning.

The extracted features include:
- recording duration
- sleep period duration
- total sleep time
- sleep latency
- REM latency
- fragmentation
- wake percentage within the sleep period
- REM proportion of total sleep time
- deep sleep proportion of total sleep time
- sleep efficiency

The next step is to scale this pipeline to multiple subjects and merge the resulting sleep features with MESA outcomes such as depression and cognitive scores.

In [ ]:
features_df.to_csv(PROCESSED_DIR / 'sleep_features_subject_001.csv', index=False)